## Poster Improvement — Instruction-Based Image Editing 
This notebook takes each poster + its TOPSIS score/issues (from the scoring notebook) and sends it to a
**free, hosted, open-weight image-editing model (Qwen-Image-Edit)** via the Hugging Face Inference Providers API.

Key idea: this is **image-to-image EDITING**, not text-to-image generation.
The model is instructed to keep the poster's layout, text, and content intact, and only fix the
specific issues detected (e.g. low contrast, cramped whitespace) — so you get an *improved* version
of the same poster, not a brand new one.

**Requirements:**
- Free Hugging Face account → https://huggingface.co/join
- Free access token → https://huggingface.co/settings/tokens (create a **Read** token; Inference Providers works with it)
- No GPU needed — inference runs on HF's servers.


In [1]:
# CELL 1 — Install dependencies
!pip install huggingface_hub pillow pandas --quiet


In [ ]:
# CELL 2 — Imports & HF token
import os, io, time, ast
import pandas as pd
from PIL import Image
from huggingface_hub import InferenceClient

# --- Paste your Hugging Face token here (starts with 'hf_') ---
# Get it free from: https://huggingface.co/settings/tokens
HF_TOKEN = os.getenv("HF_TOKEN")

os.environ["HF_TOKEN"] = HF_TOKEN
client = InferenceClient(token=HF_TOKEN)

print("Client ready.")


Client ready.


## Folder setup
- `INPUT_DIR` = folder with your original training posters (renamed poster1.jpg ... poster33.jpg)
- `SCORES_CSV` = the CSV you generated earlier (file, topsis_score, issues columns)
- `OUTPUT_DIR` = where improved posters get saved (genposter1.png ... genposter33.png)


In [4]:
# CELL 3 — Paths (EDIT THESE to match your folder structure)
INPUT_DIR   = "dataset"                 # folder containing poster1.jpg, poster2.jpg, ... poster33.jpg
SCORES_CSV  = "poster_outputs_dataset33.csv"   # CSV from the scoring notebook (must have columns: file, score, issues)
OUTPUT_DIR  = "improved_posters"        # where genposter1.png etc will be saved

os.makedirs(OUTPUT_DIR, exist_ok=True)

scores_df = pd.read_csv(SCORES_CSV)
print(scores_df.head())
print("Rows:", len(scores_df))


           file  score                                             issues  \
0  poster1.jpeg   0.03  Low contrast – text may be hard to read; Too s...   
1  poster10.png  33.21                    Too little text – add more info   
2  poster11.png  37.05                                                NaN   
3  poster12.png  13.10            Low contrast – text may be hard to read   
4  poster13.png  39.51  Too cluttered – reduce visual elements; Too ma...   

   brightness  contrast  entropy  edge_density  hue_diversity  lr_balance  \
0    251.0513   27.3651   0.5762        0.0099           27.0      0.0011   
1    214.2319   63.3316   5.0312        0.0242           21.0      0.0189   
2    175.3925   71.7123   5.9884        0.0576           30.0      0.0092   
3    236.7249   34.6599   4.8269        0.0220           27.0      0.0057   
4    134.1346   58.6366   7.6400        0.1000           42.0      0.0495   

   tb_balance  whitespace  saliency  text_density  word_count  
0      0.0

In [8]:
# CELL 4 — Build an edit instruction from the issues list
# Maps each detected issue -> a concrete, minimal edit instruction (not a redesign request)
ISSUE_TO_INSTRUCTION = {
    "Low contrast": "increase the contrast between the text and its background so the text is clearly readable",
    "Overly harsh contrast": "slightly soften the contrast so it looks less harsh",
    "Too cluttered": "reduce visual clutter by simplifying secondary decorative elements, without removing key text or the main subject",
    "Too simple": "add a little more visual interest with subtle texture or a secondary design element, without changing the composition",
    "Too many edges": "smooth out busy, jagged detail areas to make the layout feel cleaner",
    "Not enough whitespace": "add more breathing room / whitespace around the text and main elements",
    "Too many colors": "unify the color palette so it feels more consistent, keeping the dominant brand colors",
    "Dull colors": "make the colors slightly more vibrant and visually appealing",
    "Too much text": "make the text blocks feel less crowded by adjusting spacing, without deleting information",
    "Too little text": "keep the layout as is, just improve visual polish",
    "Layout imbalance": "rebalance the visual weight between left/right and top/bottom of the poster",
}
def clean_issue_text(s):
    return (str(s).replace("â€“", "-")
                   .replace("â€™", "'")
                   .replace("â€œ", '"')
                   .replace("â€", '"')
                   .strip())

issues_list = [clean_issue_text(i) for i in str(issues_raw).split(";")]
def issues_to_prompt(issues_raw):
    if pd.isna(issues_raw) or str(issues_raw).strip() in ("None", "", "[]"):
        return None
    issues_list = [i.strip() for i in str(issues_raw).split(";")]

    instructions = []
    for issue in issues_list:
        matched = False
        for key, instr in ISSUE_TO_INSTRUCTION.items():
            if key.lower() in issue.lower():
                instructions.append(instr)
                matched = True
                break
        if not matched:
            instructions.append(issue)

    if not instructions:
        return None

    prompt = (
        "This is a professional poster design that needs targeted quality improvements, not a redesign. "
        "Preserve the exact same layout, composition, subject matter, all text content and wording exactly as written, "
        "font style, logo placement, and overall visual identity. Do not add new elements, do not remove existing "
        "elements, do not change the aspect ratio, and do not reinterpret the creative concept. "
        "Apply only the following precise corrections: " + "; ".join(instructions) + ". "
        "Make these changes subtly and naturally, as a professional designer would when polishing an existing draft, "
        "so the poster still looks unmistakably like the original, just cleaner and more visually effective."
    )
    return prompt

# quick test
print(issues_to_prompt(scores_df.iloc[0]["issues"]))


This is a professional poster design that needs targeted quality improvements, not a redesign. Preserve the exact same layout, composition, subject matter, all text content and wording exactly as written, font style, logo placement, and overall visual identity. Do not add new elements, do not remove existing elements, do not change the aspect ratio, and do not reinterpret the creative concept. Apply only the following precise corrections: increase the contrast between the text and its background so the text is clearly readable; add a little more visual interest with subtle texture or a secondary design element, without changing the composition; keep the layout as is, just improve visual polish. Make these changes subtly and naturally, as a professional designer would when polishing an existing draft, so the poster still looks unmistakably like the original, just cleaner and more visually effective.


In [9]:
# CELL 5 — Call the image-editing model (Qwen-Image-Edit) for one poster
def improve_poster(image_path, prompt, retries=3, wait=10):
    """Sends the poster + edit instruction to the hosted model, returns a PIL Image."""
    with open(image_path, "rb") as f:
        img_bytes = f.read()

    for attempt in range(retries):
        try:
            edited = client.image_to_image(
                img_bytes,
                prompt=prompt,
                model="Qwen/Qwen-Image-Edit",
            )
            return edited
        except Exception as e:
            print(f"  attempt {attempt+1} failed: {e}")
            time.sleep(wait)
    raise RuntimeError(f"Failed to edit {image_path} after {retries} attempts")


In [10]:
# CELL 6 — Batch process: poster1..33 -> genposter1..33
log = []

for i in range(1, 34):
    fname_candidates = [f"poster{i}.jpg", f"poster{i}.jpeg", f"poster{i}.png"]
    src_path = None
    for cand in fname_candidates:
        p = os.path.join(INPUT_DIR, cand)
        if os.path.exists(p):
            src_path = p
            break

    if src_path is None:
        print(f"  ✗ poster{i}: file not found in {INPUT_DIR}")
        continue

    # match this poster to its row in scores_df (by filename, case-insensitive, ignoring extension)
    base = os.path.splitext(os.path.basename(src_path))[0].lower()
    row = scores_df[scores_df["file"].str.lower().str.startswith(base)]

    if row.empty:
        print(f"  ⚠ poster{i}: no score/issues row matched — skipping edit, copying original")
        Image.open(src_path).save(os.path.join(OUTPUT_DIR, f"genposter{i}.png"))
        continue

    issues_raw = row.iloc[0]["issues"]
    score = row.iloc[0]["score"]
    prompt = issues_to_prompt(issues_raw)

    if prompt is None:
        print(f"  ✓ poster{i}: no issues detected (score {score:.1f}) — keeping original as-is")
        Image.open(src_path).save(os.path.join(OUTPUT_DIR, f"genposter{i}.png"))
        log.append({"poster": f"poster{i}", "original_score": score, "prompt": None, "status": "no_change_needed"})
        continue

    print(f"  → poster{i} (score {score:.1f}): {prompt[:80]}...")
    try:
        edited_img = improve_poster(src_path, prompt)
        out_path = os.path.join(OUTPUT_DIR, f"genposter{i}.png")
        edited_img.save(out_path)
        log.append({"poster": f"poster{i}", "original_score": score, "prompt": prompt, "status": "edited"})
        print(f"    ✓ saved {out_path}")
    except Exception as e:
        print(f"    ✗ poster{i} failed: {e}")
        log.append({"poster": f"poster{i}", "original_score": score, "prompt": prompt, "status": f"failed: {e}"})

log_df = pd.DataFrame(log)
log_df.to_csv(os.path.join(OUTPUT_DIR, "edit_log.csv"), index=False)
print("\nDone. Log saved to", os.path.join(OUTPUT_DIR, "edit_log.csv"))


  → poster1 (score 0.0): This is a professional poster design that needs targeted quality improvements, n...
    ✓ saved improved_posters\genposter1.png
  → poster2 (score 29.5): This is a professional poster design that needs targeted quality improvements, n...
  attempt 1 failed: Client error '402 Payment Required' for url 'https://router.huggingface.co/fal-ai/fal-ai/qwen-image-edit?_subdomain=queue' (Request ID: Root=1-6a7d524c-74fa97f50eb209296fff3b03;9056d556-8b7b-4540-bbda-b8203a7b396f)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.
  attempt 2 failed: Client error '402 Payment Required' for url 'https://router.huggingface.co/fal-ai/fal-ai/qwen-image-edit?_subdomain=queue' (Request ID: Root=1-6a7d5257-650d25e265bddbf86b0ed0ee;f4453d41-c2a6-4a16-9bfc-aed7d13ba632)

KeyboardInterrupt: 